In [1]:
# This R environment comes with many helpful analytics packages installed
# It is defined by the kaggle/rstats Docker image: https://github.com/kaggle/docker-rstats
# For example, here's a helpful package to load

library(tidyverse) # metapackage of all tidyverse packages

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

list.files(path = "../input")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


[1] "datasets"

In [4]:
# LAB 3: CONTROL FLOW FOR DATA CLEANING
# Dataset: Heart Disease UCI
# File: Heart Disease UCI.csv

# TASK 0: LOAD AND EXPLORE DATASET

heart <- read.csv("/kaggle/input/datasets/mohamedmoharam88/heart-disease-uci/Heart Disease UCI.csv")

cat("DATASET INFORMATION\n")
cat("Rows:", nrow(heart), "\n")
cat("Columns:", ncol(heart), "\n")
cat("Column names:\n")
print(names(heart))
cat("\nFirst 6 rows:\n")
print(head(heart))
cat("\nStructure:\n")
str(heart)
cat("\nSummary:\n")
print(summary(heart))

# Check required columns
if (!all(c("trestbps", "chol") %in% names(heart))) {
  stop("Required columns 'trestbps' and 'chol' were not found.")
}

cat("\nRequired columns found successfully.\n")

# TASK 1: SIMULATE INVALID BP VALUES
# Negative BP -> -20
# Missing BP -> NA
# Extreme BP -> 350

set.seed(123)
heart_dirty <- heart
n <- nrow(heart_dirty)

negative_rows <- sample(1:n, 5)
na_rows <- sample(setdiff(1:n, negative_rows), 5)
extreme_rows <- sample(setdiff(1:n, c(negative_rows, na_rows)), 5)

heart_dirty$trestbps[negative_rows] <- -20
heart_dirty$trestbps[na_rows] <- NA
heart_dirty$trestbps[extreme_rows] <- 350

cat("\nSIMULATED INVALID VALUES\n")
cat("Negative BP:", sum(heart_dirty$trestbps < 0, na.rm = TRUE), "\n")
cat("Missing BP:", sum(is.na(heart_dirty$trestbps)), "\n")
cat("Extreme BP (>300):", sum(heart_dirty$trestbps > 300, na.rm = TRUE), "\n")

# TASK 2: BP CLEANING FUNCTION USING IF-ELSE
# Negative BP -> NA
# BP > 250 -> 250
# Valid BP -> unchanged
# NA -> NA

clean_bp <- function(bp) {
  if (is.na(bp)) {
    return(NA)
  } else if (bp < 0) {
    return(NA)
  } else if (bp > 250) {
    return(250)
  } else {
    return(bp)
  }
}

# Test cleaning function
cat("\nTESTING BP FUNCTION\n")
cat("clean_bp(-10): ")
print(clean_bp(-10))
cat("clean_bp(NA): ")
print(clean_bp(NA))
cat("clean_bp(350): ")
print(clean_bp(350))
cat("clean_bp(120): ")
print(clean_bp(120))

# Apply cleaning function
heart_dirty$trestbps_clean <- sapply(heart_dirty$trestbps, clean_bp)

# Show before and after
cat("\nBEFORE VS AFTER CLEANING\n")
comparison <- data.frame(
  Original_BP = heart_dirty$trestbps,
  Cleaned_BP = heart_dirty$trestbps_clean
)
print(head(comparison, 20))

# TASK 3: ERROR HANDLING USING tryCatch()
# Calculate mean BP while handling missing values

cat("\ntryCatch(): MEAN BP\n")

mean_bp <- tryCatch({
  mean(heart_dirty$trestbps_clean, na.rm = TRUE)
}, warning = function(w) {
  message("Warning: ", w$message)
  NA
}, error = function(e) {
  message("Error: ", e$message)
  NA
})

cat("Mean cleaned BP:", mean_bp, "\n")

# TASK 3A: CHOLESTEROL / BP RATIO
# Calculate chol / trestbps
# Handle NA, zero denominator and invalid values

calculate_ratio <- function(chol, bp) {
  tryCatch({
    if (is.na(chol) || is.na(bp)) {
      stop("NA value detected")
    }
    if (bp == 0) {
      stop("Division by zero: BP is 0")
    }
    if (chol < 0 || bp < 0) {
      stop("Invalid negative value")
    }
    chol / bp
  }, warning = function(w) {
    message("Warning: ", w$message)
    NA
  }, error = function(e) {
    message("Error: ", e$message)
    NA
  })
}

# Test ratio function
cat("\nTESTING RATIO FUNCTION\n")
cat("Valid example:\n")
print(calculate_ratio(240, 120))
cat("NA example:\n")
print(calculate_ratio(240, NA))
cat("Zero denominator example:\n")
print(calculate_ratio(240, 0))
cat("Negative value example:\n")
print(calculate_ratio(-240, 120))

# Calculate ratio for entire dataset
heart_dirty$chol_bp_ratio <- mapply(
  calculate_ratio,
  heart_dirty$chol,
  heart_dirty$trestbps_clean
)

cat("\nCHOLESTEROL/BP RATIO\n")
print(head(heart_dirty[c("chol", "trestbps_clean", "chol_bp_ratio")], 10))

# TASK 4: LOOP VS VECTORIZATION
# Detect NA, negative BP and BP > 250

# TASK 4A: FOR LOOP

cat("\nFOR LOOP\n")

invalid_loop <- c()
start_loop <- Sys.time()

for (i in 1:nrow(heart_dirty)) {
  bp <- heart_dirty$trestbps[i]
  if (is.na(bp) || bp < 0 || bp > 250) {
    invalid_loop <- c(invalid_loop, i)
  }
}

end_loop <- Sys.time()
loop_time <- end_loop - start_loop

cat("Invalid values detected:", length(invalid_loop), "\n")
cat("Loop execution time:", loop_time, "\n")

# TASK 4B: VECTORIZED OPERATION

cat("\nVECTORIZATION\n")

start_vector <- Sys.time()

invalid_vector <- which(
  is.na(heart_dirty$trestbps) |
  heart_dirty$trestbps < 0 |
  heart_dirty$trestbps > 250
)

end_vector <- Sys.time()
vector_time <- end_vector - start_vector

cat("Invalid values detected:", length(invalid_vector), "\n")
cat("Vectorized execution time:", vector_time, "\n")

# TASK 4C: PERFORMANCE COMPARISON

cat("\nPERFORMANCE COMPARISON\n")
cat("For loop time:", loop_time, "\n")
cat("Vectorized time:", vector_time, "\n")
cat("Same invalid rows detected:", identical(invalid_loop, invalid_vector), "\n")

performance <- data.frame(
  Method = c("For Loop", "Vectorization"),
  Invalid_Values = c(length(invalid_loop), length(invalid_vector)),
  Execution_Time = c(as.numeric(loop_time), as.numeric(vector_time))
)

cat("\nPerformance Table:\n")
print(performance)

# TASK 5: DATA VALIDATION
# Count missing values
# Find minimum, maximum, mean and median
# Check for negative values
# Check for values greater than 250

cat("\nDATA VALIDATION\n")

missing_count <- sum(is.na(heart_dirty$trestbps_clean))
min_bp <- min(heart_dirty$trestbps_clean, na.rm = TRUE)
max_bp <- max(heart_dirty$trestbps_clean, na.rm = TRUE)
mean_bp <- mean(heart_dirty$trestbps_clean, na.rm = TRUE)
median_bp <- median(heart_dirty$trestbps_clean, na.rm = TRUE)

negative_check <- any(heart_dirty$trestbps_clean < 0, na.rm = TRUE)
above_250_check <- any(heart_dirty$trestbps_clean > 250, na.rm = TRUE)

cat("Missing BP:", missing_count, "\n")
cat("Minimum BP:", min_bp, "\n")
cat("Maximum BP:", max_bp, "\n")
cat("Mean BP:", mean_bp, "\n")
cat("Median BP:", median_bp, "\n")
cat("Negative values present:", negative_check, "\n")
cat("Values >250 present:", above_250_check, "\n")

# TASK 5A: VALIDATION TABLE

validation <- data.frame(
  Missing_Values = missing_count,
  Minimum_BP = min_bp,
  Maximum_BP = max_bp,
  Mean_BP = mean_bp,
  Median_BP = median_bp,
  Negative_Values = negative_check,
  Values_Above_250 = above_250_check
)

cat("\nFINAL VALIDATION TABLE\n")
print(validation)

# TASK 6: SAVE CLEANED DATASET

write.csv(heart_dirty, "cleaned_heart_data.csv", row.names = FALSE)

cat("\nCleaned dataset saved as: cleaned_heart_data.csv\n")

# TASK 7: FINAL SUMMARY

cat("\nFINAL LAB SUMMARY\n")
cat("Rows:", nrow(heart), "\n")
cat("Columns:", ncol(heart), "\n")
cat("Missing cleaned BP:", missing_count, "\n")
cat("Minimum cleaned BP:", min_bp, "\n")
cat("Maximum cleaned BP:", max_bp, "\n")
cat("Mean cleaned BP:", mean_bp, "\n")
cat("Median cleaned BP:", median_bp, "\n")
cat("Negative values remaining:", negative_check, "\n")
cat("Values >250 remaining:", above_250_check, "\n")
cat("Output file: cleaned_heart_data.csv\n")

DATASET INFORMATION
Rows: 297 
Columns: 14 
Column names:
 [1] "age"       "sex"       "cp"        "trestbps"  "chol"      "fbs"      
 [7] "restecg"   "thalach"   "exang"     "oldpeak"   "slope"     "ca"       
[13] "thal"      "condition"

First 6 rows:
  age sex cp trestbps chol fbs restecg thalach exang oldpeak slope ca thal
1  69   1  0      160  234   1       2     131     0     0.1     1  1    0
2  69   0  0      140  239   0       0     151     0     1.8     0  2    0
3  66   0  0      150  226   0       0     114     0     2.6     2  0    0
4  65   1  0      138  282   1       2     174     0     1.4     1  1    0
5  64   1  0      110  211   0       2     144     1     1.8     1  0    0
6  64   1  0      170  227   0       2     155     0     0.6     1  0    2
  condition
1         0
2         0
3         0
4         1
5         0
6         0

Structure:
'data.frame':	297 obs. of  14 variables:
 $ age      : int  69 69 66 65 64 64 63 61 60 59 ...
 $ sex      : int  1 0 0 1 1 

Error: NA value detected



[1] NA
Zero denominator example:


Error: Division by zero: BP is 0



[1] NA
Negative value example:


Error: Invalid negative value



[1] NA


Error: NA value detected

Error: NA value detected

Error: NA value detected

Error: NA value detected

Error: NA value detected

Error: NA value detected

Error: NA value detected

Error: NA value detected

Error: NA value detected

Error: NA value detected




CHOLESTEROL/BP RATIO
   chol trestbps_clean chol_bp_ratio
1   234            160      1.462500
2   239            140      1.707143
3   226            150      1.506667
4   282            138      2.043478
5   211            110      1.918182
6   227            170      1.335294
7   233            145      1.606897
8   234            134      1.746269
9   240            150      1.600000
10  270            178      1.516854

FOR LOOP
Invalid values detected: 15 
Loop execution time: 0.009836435 

VECTORIZATION
Invalid values detected: 15 
Vectorized execution time: 0.001780033 

PERFORMANCE COMPARISON
For loop time: 0.009836435 
Vectorized time: 0.001780033 
Same invalid rows detected: TRUE 

Performance Table:
         Method Invalid_Values Execution_Time
1      For Loop             15    0.009836435
2 Vectorization             15    0.001780033

DATA VALIDATION
Missing BP: 10 
Minimum BP: 94 
Maximum BP: 250 
Mean BP: 133.7596 
Median BP: 130 
Negative values present: FALSE 
Values 